In [1]:
import torch
import torch.nn as nn
from transformers import T5ForConditionalGeneration, T5Tokenizer, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from torch.optim import AdamW
import math

In [2]:
class TitleDataset(Dataset):
    def __init__(self, split, tokenizer, max_input_len=512, max_target_len=64):
        self.tokenizer=tokenizer
        self.max_input_len= max_input_len
        self.max_target_len=max_target_len
        raw= load_dataset("cnn_dailymail", "3.0.0", split=split)
        self.articles = raw["article"]
        self.titles = raw["highlights"]
    def __len__(self):
        return len(self.articles)
    def __getitem__(self, idx):
        input_text= "summarize: " + self.articles[idx]
        target_text = self.titles[idx]
        inputs = self.tokenizer(
            input_text,
            max_length=self.max_input_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        targets=self.tokenizer(
            target_text, 
            max_length=self.max_target_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        labels = targets["input_ids"].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": labels
        }


def get_dataloader(split, tokenizer, batch_size=8, shuffle=True):
    dataset = TitleDataset(split=split, tokenizer=tokenizer)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

In [3]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_key_padding_mask=None):
        
        attn_out, _ = self.attn(x, x, x, key_padding_mask=src_key_padding_mask)
        x = self.norm1(x + self.dropout(attn_out))

        
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))

        return x


class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, src_key_padding_mask=None):
        for layer in self.layers:
            x = layer(x, src_key_padding_mask)
        return x

In [4]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_out, tgt_mask=None, tgt_key_padding_mask=None):
        
        attn_out, _ = self.masked_attn(x, x, x, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask)
        x = self.norm1(x + self.dropout(attn_out))

       
        attn_out, _ = self.cross_attn(x, encoder_out, encoder_out)
        x = self.norm2(x + self.dropout(attn_out))

       
        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout(ff_out))

        return x


class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, encoder_out, tgt_mask=None, tgt_key_padding_mask=None):
        for layer in self.layers:
            x = layer(x, encoder_out, tgt_mask, tgt_key_padding_mask)
        return x

In [5]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_heads=8, d_ff=512, 
                 num_layers=4, max_seq_len=512, dropout=0.1):
        super().__init__()

        # embeddings
        self.src_embedding = nn.Embedding(vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = self._make_pe(max_seq_len, d_model)

        # encoder and decoder
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)

        # final projection to vocab
        self.output_proj = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def _make_pe(self, max_seq_len, d_model):
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return nn.Parameter(pe.unsqueeze(0), requires_grad=False)

    def make_tgt_mask(self, tgt):
        seq_len = tgt.shape[1]
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        return mask.to(tgt.device)

    def forward(self, src, tgt, src_key_padding_mask=None, tgt_key_padding_mask=None):
        
        src_seq_len = src.shape[1]
        tgt_seq_len = tgt.shape[1]

        src_x = self.dropout(self.src_embedding(src) * math.sqrt(self.d_model) 
                             + self.positional_encoding[:, :src_seq_len, :])
        tgt_x = self.dropout(self.tgt_embedding(tgt) * math.sqrt(self.d_model) 
                             + self.positional_encoding[:, :tgt_seq_len, :])

        
        encoder_out = self.encoder(src_x, src_key_padding_mask)

       
        tgt_mask = self.make_tgt_mask(tgt)
        decoder_out = self.decoder(tgt_x, encoder_out, tgt_mask, tgt_key_padding_mask)

        # project to vocab size
        return self.output_proj(decoder_out)

In [6]:
def train(num_epochs=7, batch_size=32, lr=5e-4):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = T5Tokenizer.from_pretrained("t5-small")
    vocab_size = tokenizer.vocab_size

    model = Transformer(
        vocab_size=vocab_size,
        d_model=128,
        num_heads=4,
        d_ff=256,
        num_layers=2
    ).to(device)

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    train_loader = get_dataloader("train", tokenizer, batch_size=batch_size)

    optimizer = AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)

    total_steps = len(train_loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=total_steps // 10,
        num_training_steps=total_steps
    )

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0

        for batch_idx, batch in enumerate(train_loader):
            src = batch["input_ids"].to(device)
            tgt = batch["labels"].to(device)

            tgt_input = tgt[:, :-1].clone()
            tgt_input[tgt_input == -100] = tokenizer.pad_token_id
            tgt_label = tgt[:, 1:]

            src_padding_mask = (src == tokenizer.pad_token_id).to(device)
            tgt_padding_mask = (tgt_input == tokenizer.pad_token_id).to(device)

            optimizer.zero_grad()

            logits = model(src, tgt_input, src_padding_mask, tgt_padding_mask)

            loss = criterion(
                logits.reshape(-1, vocab_size),
                tgt_label.reshape(-1)
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch+1} | Batch {batch_idx} | Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)
        print(f"--- Epoch {epoch+1} complete | Avg Loss: {avg_loss:.4f} ---")
        torch.save(
            model.module.state_dict() if hasattr(model, 'module') else model.state_dict(),
            f"/kaggle/working/transformer_epoch_{epoch+1}.pt"
        )
        print(f"Checkpoint saved for epoch {epoch+1}")

    torch.save(
        model.module.state_dict() if hasattr(model, 'module') else model.state_dict(),
        "/kaggle/working/transformer_title_generator.pt"
    )
    print("Done. Model saved.")
    return model, tokenizer

In [7]:
def generate(model, tokenizer, num_examples=3, max_title_len=64):
    device = next(model.parameters()).device
    m = model.module if hasattr(model, 'module') else model
    m.eval()

    dataset = load_dataset("cnn_dailymail", "3.0.0", split="test")

    for i in range(num_examples):
        article = dataset["article"][i]
        real_title = dataset["highlights"][i]

        inputs = tokenizer(
            "summarize: " + article,
            max_length=512,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        src = inputs["input_ids"]
        src_padding_mask = (src == tokenizer.pad_token_id).to(device)
        generated = torch.tensor([[tokenizer.pad_token_id]], device=device)

        with torch.no_grad():
            for _ in range(max_title_len):
                logits = m(src, generated, src_padding_mask)
                next_logits = logits[:, -1, :]

                for token_id in generated[0]:
                    next_logits[0, token_id] /= 1.5

                next_token = next_logits.argmax(dim=-1, keepdim=True)
                generated = torch.cat([generated, next_token], dim=1)
                if next_token.item() == tokenizer.eos_token_id:
                    break

        title = tokenizer.decode(generated[0], skip_special_tokens=True)

        print("=" * 60)
        print(f"ARTICLE (first 300 chars):\n{article[:300]}...")
        print(f"\nREAL TITLE:\n{real_title}")
        print(f"\nGENERATED TITLE:\n{title}")
        print("=" * 60)

In [8]:
model, tokenizer = train(num_epochs=6, batch_size=32, lr=5e-4)
generate(model, tokenizer)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Using 2 GPUs


README.md: 0.00B [00:00, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Epoch 1 | Batch 0 | Loss: 10.5034
Epoch 1 | Batch 100 | Loss: 10.3954
Epoch 1 | Batch 200 | Loss: 9.8668
Epoch 1 | Batch 300 | Loss: 9.2861
Epoch 1 | Batch 400 | Loss: 8.7369
Epoch 1 | Batch 500 | Loss: 8.1547
Epoch 1 | Batch 600 | Loss: 7.5816
Epoch 1 | Batch 700 | Loss: 7.1714
Epoch 1 | Batch 800 | Loss: 6.9348
Epoch 1 | Batch 900 | Loss: 6.8879
Epoch 1 | Batch 1000 | Loss: 6.7011
Epoch 1 | Batch 1100 | Loss: 6.6417
Epoch 1 | Batch 1200 | Loss: 6.6406
Epoch 1 | Batch 1300 | Loss: 6.6677
Epoch 1 | Batch 1400 | Loss: 6.6143
Epoch 1 | Batch 1500 | Loss: 6.4950
Epoch 1 | Batch 1600 | Loss: 6.6518
Epoch 1 | Batch 1700 | Loss: 6.4905
Epoch 1 | Batch 1800 | Loss: 6.5539
Epoch 1 | Batch 1900 | Loss: 6.6238
Epoch 1 | Batch 2000 | Loss: 6.4181
Epoch 1 | Batch 2100 | Loss: 6.4733
Epoch 1 | Batch 2200 | Loss: 6.2909
Epoch 1 | Batch 2300 | Loss: 6.3566
Epoch 1 | Batch 2400 | Loss: 6.2537
Epoch 1 | Batch 2500 | Loss: 6.4992
Epoch 1 | Batch 2600 | Loss: 6.3278
Epoch 1 | Batch 2700 | Loss: 6.1816
Ep